# Quickstart with Milvus Lite

向量是神经网络模型的输出数据格式，能够有效编码信息，在知识库、语义搜索、检索增强生成（RAG）等人工智能应用中发挥关键作用。
Milvus 是一个开源的向量数据库，适用于各种规模的人工智能应用，无论是运行 Jupyter 笔记本中的演示聊天机器人，还是构建可服务数十亿用户的网络级搜索系统。在本指南中，我们将带您快速完成 Milvus 在本地的部署，并使用 Python 客户端库生成、存储和查询向量。

In [1]:
#!uv pip install pymilvus

In [2]:
import sys
import os

print(sys.executable)
os.environ['HF_HOME'] = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = os.path.join(os.getcwd(), 'models')

F:\Teewon\Milvue\.venv\Scripts\python.exe


## Set Up Vector Database

设置向量数据库

In [3]:
#!uv pip install milvus-lite

In [4]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")

## Create a Collection(表)

在 Milvus 中，我们需要一个集合来存储向量及其关联的元数据。你可以将其想象为传统 SQL 数据库中的一个表。创建集合时，可以定义模式和索引参数，以配置向量的规格，例如维度、索引类型和距离度量。此外，还有复杂的概念用于优化索引，以提升向量搜索性能。目前，我们先专注于基本功能，尽可能使用默认值。至少，你只需要设置集合名称以及集合中向量字段的维度即可。

In [5]:
if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")
client.create_collection(
    collection_name="demo_collection",
    dimension=768,  # The vectors we will use in this demo has 768 dimensions
)

在上述设置中，
- 主键和向量字段使用其默认名称（“id”和“vector”）。
- 度量类型（向量距离定义）设置为默认值（COSINE）。
- 主键字段接受整数，且不会自动递增（即不使用自动ID功能）。您也可以按照以下说明，正式定义集合的模式。

## Prepare Data

在本指南中，我们使用向量对文本进行语义搜索。需要通过下载嵌入模型来生成文本的向量。这可以通过 pymilvus[model] 库中的实用函数轻松完成。

### Represent text with vectors

首先，安装模型库。该包包含 PyTorch 等必要的机器学习工具。如果您的本地环境从未安装过 PyTorch，下载该包可能需要一些时间。

In [6]:
# !uv pip install "pymilvus[model]"

使用默认模型生成向量嵌入。Milvus 期望数据以字典列表的形式插入，其中每个字典表示一条数据记录，称为实体。

In [7]:
from pymilvus.model.dense import SentenceTransformerEmbeddingFunction
# 如果连接到 https://huggingface.co/ 失败，请取消注释以下路径：
# import os
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 这将下载一个小型嵌入模型“paraphrase-albert-small-v2”（约50MB）。
embedding_fn=SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

# 要搜索的文本字符串
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

vectors=embedding_fn.encode_documents(docs)
# 输出向量有768个维度，与我们刚刚创建的Collection(表)相匹配
print("Dim:", embedding_fn.dim, vectors[0].shape)# Dim: 768 (768,)

# 每个实体都有ID、向量表示、原始文本以及一个我们用于后续演示元数据过滤的主体标签
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dim: 768 (768,)
Data has 3 entities, each with fields:  dict_keys(['id', 'vector', 'text', 'subject'])
Vector dim: 768


F:\Teewon\Milvue\.venv\Lib\site-packages\pymilvus\model\dense\sentence_transformer.py:46: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


### [Alternatively] 使用随机向量的虚假表示

如果由于网络问题无法下载模型，作为替代方案，你可以使用随机向量来表示文本，仍然可以完成示例。但请注意，搜索结果不会反映语义相似性，因为这些向量是虚拟的。

```
import random

# Text strings to search from.
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]
# Use fake representation with random vectors (768 dimension).
vectors = [[random.uniform(-1, 1) for _ in range(768)] for _ in docs]
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))
```

## Insert Data

让我们将数据插入到Collection中：

In [9]:
res=client.insert(collection_name="demo_collection",data=data)

print(res)

{'insert_count': 3, 'ids': [0, 1, 2]}
